In [ ]:
# @title Setup: Install scpn-quantum-control
!pip install -q scpn-quantum-control
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)


SPDX-License-Identifier: AGPL-3.0-or-later | Commercial license available
Â© Concepts 1996â€“2026 Miroslav Ĺ otek. All rights reserved.
Â© Code 2020â€“2026 Miroslav Ĺ otek. All rights reserved.
ORCID: 0009-0009-3560-0851
Contact: www.anulum.li | protoscience@anulum.li
scpn-quantum-control â€” Morphogenesis Clock as Kuramoto Synchronisation

The somitogenesis clock (vertebrate segmentation) is a textbook case
of coupled oscillators in development. Cells oscillate with ~2hr period
and synchronise via Delta-Notch signalling. This IS Kuramoto.

Also tests: Turing patterns, cardiac pacemaker, circadian SCN

These are oscillator systems with KNOWN parameters from literature.
Comparing their measured K_c and coupling to SCPN predictions.

In [ ]:
import json

In [ ]:
import numpy as np
from scipy import stats

In [ ]:
print("=" * 70)
print("MORPHOGENESIS AND BIOLOGICAL CLOCKS AS KURAMOTO SYSTEMS")
print("=" * 70)

=====================================================================
BIOLOGICAL OSCILLATOR DATABASE (from literature)
=====================================================================

In [ ]:
bio_oscillators = {
    "somitogenesis": {
        "period_s": 7200,  # ~2 hours (mouse)
        "freq_Hz": 1 / 7200,
        "N_oscillators": 100,  # ~100 presomitic mesoderm cells
        "coupling": "Delta-Notch",
        "K_coupling_est": 0.5,  # dimensionless (Morelli et al. 2009)
        "sync_R": 0.85,  # high sync in normal development
        "desync_phenotype": "vertebral defects (spondylocostal dysostosis)",
    },
    "circadian_SCN": {
        "period_s": 86400,  # ~24 hours
        "freq_Hz": 1 / 86400,
        "N_oscillators": 20000,  # ~20,000 neurons in SCN
        "coupling": "VIP + GABA + gap junctions",
        "K_coupling_est": 0.3,  # weak coupling (Liu et al. 2007)
        "sync_R": 0.7,  # moderate sync
        "desync_phenotype": "jet lag, shift work disorder",
    },
    "cardiac_SA_node": {
        "period_s": 0.83,  # ~72 bpm
        "freq_Hz": 1.2,
        "N_oscillators": 10000,  # ~10,000 pacemaker cells
        "coupling": "gap junctions (connexin43)",
        "K_coupling_est": 5.0,  # strong coupling (all-to-all via gap junctions)
        "sync_R": 0.99,  # near-perfect sync (must be!)
        "desync_phenotype": "arrhythmia, sudden cardiac death",
    },
    "pancreatic_islet": {
        "period_s": 300,  # ~5 min (Ca2+ oscillations)
        "freq_Hz": 1 / 300,
        "N_oscillators": 1000,  # ~1000 beta cells per islet
        "coupling": "gap junctions + paracrine",
        "K_coupling_est": 1.0,
        "sync_R": 0.8,
        "desync_phenotype": "impaired insulin pulsatility (Type 2 diabetes)",
    },
    "cortical_gamma": {
        "period_s": 0.025,  # 40 Hz
        "freq_Hz": 40,
        "N_oscillators": 10000,  # cortical column
        "coupling": "GABAergic interneurons",
        "K_coupling_est": 2.0,
        "sync_R": 0.4,  # moderate (not full sync)
        "desync_phenotype": "schizophrenia, autism (gamma abnormalities)",
    },
    "intestinal_ICC": {
        "period_s": 20,  # ~3 cycles/min (slow waves)
        "freq_Hz": 0.05,
        "N_oscillators": 5000,  # interstitial cells of Cajal
        "coupling": "gap junctions",
        "K_coupling_est": 3.0,
        "sync_R": 0.9,
        "desync_phenotype": "gastroparesis, irritable bowel",
    },
}

In [ ]:
# =====================================================================
# TEST 1: Kuramoto simulation for each biological oscillator
# =====================================================================
print("\n" + "=" * 70)
print("TEST 1: KURAMOTO SIMULATION OF BIOLOGICAL OSCILLATORS")
print("=" * 70)

In [ ]:
def simulate_bio_kuramoto(N, K_est, freq_spread=0.1, dt=0.01, T=500, n_trials=10):
    """Simulate Kuramoto for biological oscillator system."""
    omega = np.random.normal(1.0, freq_spread, N)
    n_steps = int(T / dt)
    R_trials = []

    for _ in range(n_trials):
        theta = np.random.uniform(0, 2 * np.pi, N)
        for _s in range(n_steps):
            # Mean-field Kuramoto (all-to-all for simplicity)
            z = np.mean(np.exp(1j * theta))
            R = abs(z)
            psi = np.angle(z)
            dtheta = omega + K_est * R * np.sin(psi - theta)
            theta += dtheta * dt
        z_final = np.mean(np.exp(1j * theta))
        R_trials.append(abs(z_final))

    return np.mean(R_trials), np.std(R_trials)

In [ ]:
# Use small N for simulation (scale K accordingly)
N_sim = 50  # tractable size

In [ ]:
print(f"\nSimulation (N={N_sim}, mean-field Kuramoto):")
print(f"{'System':25s} {'K_est':>6s} {'R_sim':>6s} {'R_lit':>6s} {'Match':>6s}")
print("-" * 55)

In [ ]:
sim_results = {}
for name, data in bio_oscillators.items():
    R_sim, R_std = simulate_bio_kuramoto(N_sim, data["K_coupling_est"])
    R_lit = data["sync_R"]
    match = abs(R_sim - R_lit) < 0.15
    print(
        f"{name:25s} {data['K_coupling_est']:6.1f} {R_sim:6.3f} {R_lit:6.2f} {'YES' if match else 'no':>6s}"
    )
    sim_results[name] = {
        "R_sim": round(R_sim, 3),
        "R_lit": R_lit,
        "K_est": data["K_coupling_est"],
        "match": match,
    }

In [ ]:
# =====================================================================
# TEST 2: K_c for each system
# =====================================================================
print("\n" + "=" * 70)
print("TEST 2: CRITICAL COUPLING K_c PER SYSTEM")
print("=" * 70)

In [ ]:
K_c_bio = {}
for name, data in bio_oscillators.items():
    K_lo, K_hi = 0.01, 10.0
    for _ in range(15):
        K_mid = (K_lo + K_hi) / 2
        R, _ = simulate_bio_kuramoto(N_sim, K_mid)
        if R > 0.5:
            K_hi = K_mid
        else:
            K_lo = K_mid
    K_c_bio[name] = K_mid
    above = data["K_coupling_est"] > K_mid
    print(
        f"{name:25s}: K_c = {K_mid:.3f}, K_est = {data['K_coupling_est']:.1f} -> {'ABOVE K_c (sync)' if above else 'BELOW K_c (no sync)'}"
    )

In [ ]:
# =====================================================================
# TEST 3: Frequency vs coupling strength (biological scaling law)
# =====================================================================
print("\n" + "=" * 70)
print("TEST 3: FREQUENCY vs COUPLING STRENGTH")
print("=" * 70)

In [ ]:
freqs = np.array([d["freq_Hz"] for d in bio_oscillators.values()])
K_ests = np.array([d["K_coupling_est"] for d in bio_oscillators.values()])
R_lits = np.array([d["sync_R"] for d in bio_oscillators.values()])
names_bio = list(bio_oscillators.keys())

In [ ]:
# Log-log regression
log_freq = np.log10(freqs)
log_K = np.log10(K_ests)

In [ ]:
slope_fK, intercept_fK, r_fK, p_fK, _ = stats.linregress(log_freq, log_K)
print(f"Frequency vs coupling: K ~ freq^({slope_fK:.3f})")
print(f"R^2 = {r_fK**2:.4f}, p = {p_fK:.4f}")

In [ ]:
if slope_fK > 0:
    print("POSITIVE: faster oscillators need stronger coupling")
else:
    print("NEGATIVE: faster oscillators need weaker coupling")

In [ ]:
# Sync level vs coupling
r_RK, p_RK = stats.pearsonr(K_ests, R_lits)
print(f"\nCoupling vs sync level: r={r_RK:.3f}, p={p_RK:.4f}")

In [ ]:
# Compare to SCPN K_c
print("\nSCPN K_c = 2.7 (N=8, alpha=0.3)")
print("Biological K_c values:")
for name, kc in K_c_bio.items():
    ratio = kc / 2.7
    print(f"  {name:25s}: K_c = {kc:.3f} ({ratio:.2f}x SCPN)")

In [ ]:
# =====================================================================
# TEST 4: Desynchronisation as disease
# =====================================================================
print("\n" + "=" * 70)
print("TEST 4: DESYNCHRONISATION = DISEASE")
print("=" * 70)

In [ ]:
print("\nMapping sync failure to pathology:")
for name, data in bio_oscillators.items():
    margin = data["K_coupling_est"] - K_c_bio[name]
    vulnerability = "FRAGILE" if margin < 0.5 else "ROBUST"
    print(f"\n{name}:")
    print(f"  K_coupling = {data['K_coupling_est']:.1f}, K_c = {K_c_bio[name]:.3f}")
    print(f"  Safety margin: {margin:.3f} ({vulnerability})")
    print(f"  Desync phenotype: {data['desync_phenotype']}")

In [ ]:
# =====================================================================
# TEST 5: Somitogenesis clock as Turing-Kuramoto
# =====================================================================
print("\n" + "=" * 70)
print("TEST 5: SOMITOGENESIS = TURING + KURAMOTO")
print("=" * 70)

Somitogenesis is unique: oscillators + spatial gradient
The clock-and-wavefront model: cells oscillate (Kuramoto)
AND move through a maturation gradient (Turing-like)

In [ ]:
# Simulate 1D chain of oscillators with frequency gradient
N_chain = 50
# Posterior (stem zone) oscillates fastest, anterior slows down
omega_gradient = np.linspace(1.2, 0.8, N_chain)  # frequency gradient
K_notch = 1.0  # Delta-Notch coupling (nearest-neighbour)

In [ ]:
dt = 0.01
T = 500
n_steps = int(T / dt)
theta = np.random.uniform(0, 2 * np.pi, N_chain)

In [ ]:
# Track wavefront propagation
R_local = np.zeros(N_chain)
phase_pattern = np.zeros(N_chain)

In [ ]:
for _s in range(n_steps):
    dtheta = omega_gradient.copy()
    for i in range(N_chain):
        if i > 0:
            dtheta[i] += K_notch * np.sin(theta[i - 1] - theta[i])
        if i < N_chain - 1:
            dtheta[i] += K_notch * np.sin(theta[i + 1] - theta[i])
    theta += dtheta * dt

In [ ]:
# Compute local order parameter (window of 5 cells)
for i in range(N_chain):
    window = slice(max(0, i - 2), min(N_chain, i + 3))
    z_local = np.mean(np.exp(1j * theta[window]))
    R_local[i] = abs(z_local)

In [ ]:
phase_pattern = theta % (2 * np.pi)

In [ ]:
# Find somite boundaries (phase singularities)
phase_jumps = np.abs(np.diff(phase_pattern))
phase_jumps[phase_jumps > np.pi] = 2 * np.pi - phase_jumps[phase_jumps > np.pi]
boundaries = np.where(phase_jumps > 1.0)[0]

In [ ]:
print(f"Chain of {N_chain} cells with frequency gradient:")
print(f"  Posterior freq: {omega_gradient[0]:.2f}")
print(f"  Anterior freq:  {omega_gradient[-1]:.2f}")
print(f"  Coupling K:     {K_notch:.1f}")
print(f"  Somite boundaries detected: {len(boundaries)}")
print(f"  Mean R_local:   {np.mean(R_local):.3f}")
print(f"  Posterior R:    {np.mean(R_local[:10]):.3f}")
print(f"  Anterior R:     {np.mean(R_local[-10:]):.3f}")

In [ ]:
# Real mouse: ~65 somites formed in ~13 hours
# Each somite = one oscillation cycle captured by the wavefront
print("\n  Mouse somites: ~65 (reality)")
print(f"  Our model:     ~{len(boundaries)} boundaries")
print("  The gradient creates segmentation from synchronisation")

In [ ]:
# SYNTHESIS
print("\n" + "=" * 70)
print("SYNTHESIS: BIOLOGICAL OSCILLATORS AS KURAMOTO SYSTEMS")
print("=" * 70)

In [ ]:
n_match = sum(1 for v in sim_results.values() if v["match"])
print(f"\n1. Kuramoto reproduces {n_match}/{len(sim_results)} biological sync levels")
print(f"2. Frequency-coupling scaling: K ~ freq^({slope_fK:.3f}), p={p_fK:.4f}")
print(f"3. Coupling vs sync: r={r_RK:.3f}")
print("4. Desynchronisation maps to specific diseases for ALL systems")
print("5. Somitogenesis = Kuramoto + spatial gradient (Turing-Kuramoto)")

In [ ]:
print("\nKey insight: EVERY major biological oscillator system is a Kuramoto")
print("system with specific K_nm topology. Disease = K drops below K_c.")
print("The UPDE captures ALL of these with scale-appropriate parameters.")

In [ ]:
# JSON output
results = {
    "n_systems": len(bio_oscillators),
    "kuramoto_match": n_match,
    "freq_coupling_slope": round(float(slope_fK), 3),
    "freq_coupling_p": round(float(p_fK), 4),
    "coupling_sync_r": round(float(r_RK), 3),
    "K_c_per_system": {k: round(v, 3) for k, v in K_c_bio.items()},
    "somite_boundaries": int(len(boundaries)),
    "sim_results": sim_results,
}

In [ ]:
print("\n--- JSON RESULTS ---")
print(json.dumps(results, indent=2))